<a href="https://colab.research.google.com/github/luciacardozo472/TRABAJOP_IA/blob/main/sales_agent_colab_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 Agente de Análisis de Ventas
**LangChain + Pandas + Mistral Small — Sin RAG**

In [ ]:
# Celda 1 — Instalación (una sola línea, instala todo lo necesario)
!pip install -q langchain-experimental langchain-mistralai pandas tabulate

In [ ]:
# Celda 2 — API Key de Mistral
import os

# OPCIÓN A: pegar la key directo
# os.environ["MISTRAL_API_KEY"] = "tu_api_key_aqui"

# OPCIÓN B: Secrets de Colab (recomendado)
# Menú izquierdo 🔑 → agregar MISTRAL_API_KEY
from google.colab import userdata
os.environ["MISTRAL_API_KEY"] = userdata.get("MISTRAL_API_KEY")

print('✅ API Key configurada')

In [ ]:
# Celda 3 — Subir y cargar el CSV
import pandas as pd
from google.colab import files

uploaded = files.upload()
filename = list(uploaded.keys())[0]

df = pd.read_csv(filename)
df['orderdate'] = pd.to_datetime(df['orderdate'], errors='coerce')
df['month'] = df['orderdate'].dt.month
df['year']  = df['orderdate'].dt.year

print(f'✅ Dataset cargado: {len(df):,} filas × {len(df.columns)} columnas')
df.head()

In [ ]:
# Celda 4 — Crear el agente
import warnings
warnings.filterwarnings('ignore')

from langchain_mistralai import ChatMistralAI
from langchain_experimental.agents import create_pandas_dataframe_agent

llm = ChatMistralAI(
    model='mistral-small-latest',
    mistral_api_key=os.environ['MISTRAL_API_KEY'],
    temperature=0,
)

agente = create_pandas_dataframe_agent(
    llm=llm,
    df=df,
    verbose=True,
    allow_dangerous_code=True,
    max_iterations=10,
    handle_parsing_errors=True,
)

print('✅ Agente listo — Mistral Small + LangChain + Pandas')

In [ ]:
# Celda 5 — Hacer una consulta
pregunta = '¿Cuál es el total de ventas por país?'

respuesta = agente.invoke({'input': pregunta})
print('\n🟢 Respuesta:', respuesta['output'])

In [ ]:
# Celda 6 — Demo automática (8 preguntas)
preguntas_demo = [
    '¿Cuántas filas y columnas tiene el dataset?',
    '¿Cuál es el total de ventas (sales) por país (country)?',
    '¿Cuál es el producto más vendido por cantidad (quantityordered)?',
    '¿Cuál es el promedio de ventas por línea de producto (productline)?',
    '¿Cuántos pedidos hay en cada estado (status)?',
    '¿Cuál es el cliente con mayor volumen total de ventas?',
    '¿Cuál es el mes con más ventas?',
    '¿Cuántos pedidos son de tamaño Large?',
]

for i, pregunta in enumerate(preguntas_demo, 1):
    print(f'\n{"="*60}')
    print(f'[{i}/{len(preguntas_demo)}] {pregunta}')
    print('='*60)
    try:
        respuesta = agente.invoke({'input': pregunta})
        print(f'🟢 {respuesta["output"]}')
    except Exception as e:
        print(f'⚠️ Error: {e}')

CON RAG

Instalación de librerías RAG

In [ ]:
!pip install -q faiss-cpu langchain-community sentence-transformers

 Convertir el CSV en documentos y construir el índice FAISS

In [ ]:
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

# Convertir cada fila del CSV en un documento de texto
def fila_a_texto(row):
    return (
        f"Orden {row['ordernumber']}: cliente {row['customername']} de {row['country']}, "
        f"producto {row['productcode']} ({row['productline']}), "
        f"cantidad {row['quantityordered']}, precio unitario {row['priceeach']}, "
        f"venta total {row['sales']}, estado {row['status']}, "
        f"tamaño del trato {row['dealsize']}, fecha {row['orderdate']}."
    )

documentos = [
    Document(page_content=fila_a_texto(row), metadata={'ordernumber': row['ordernumber']})
    for _, row in df.iterrows()
]

print(f'📄 {len(documentos)} documentos creados')

# Modelo de embeddings local (no necesita API key)
embeddings = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')

# Construir el índice FAISS
print('⏳ Vectorizando documentos con FAISS...')
vectorstore = FAISS.from_documents(documentos, embeddings)

print('✅ Índice FAISS construido')

 Crear la cadena RAG:

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# Recuperador: trae los 5 fragmentos más relevantes
retriever = vectorstore.as_retriever(search_kwargs={'k': 5})

# Prompt RAG
prompt_rag = ChatPromptTemplate.from_template("""
Usá el siguiente contexto extraído de datos de ventas para responder la pregunta.
Si no encontrás la información en el contexto, decilo claramente.

Contexto:
{context}

Pregunta: {question}

Respuesta:""")

# Función para formatear los documentos recuperados
def formatear_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Cadena RAG con LCEL
cadena_rag = (
    {"context": retriever | formatear_docs, "question": RunnablePassthrough()}
    | prompt_rag
    | llm
    | StrOutputParser()
)

print('✅ Cadena RAG lista')

Consulta con RAG:

In [ ]:
pregunta = '¿Cuáles son los pedidos de Euro Shopping Channel?'

respuesta = cadena_rag.invoke(pregunta)

print('🟢 Respuesta (Con RAG):')
print(respuesta)


Comparación Sin RAG vs Con RAG:

In [ ]:

pregunta = '¿Qué pedidos tienen estado Cancelled?'

print('='*60)
print('🅰️  SIN RAG')
print('='*60)
r_sin = agente.invoke({'input': pregunta})
print('🟢', r_sin['output'])

print('\n' + '='*60)
print('🅱️  CON RAG')
print('='*60)
r_con = cadena_rag.invoke(pregunta)
print('🟢', r_con)

Chat interactivo con RAG

In [ ]:
from IPython.display import display, HTML
import ipywidgets as widgets

# Historial del chat
historial = []

# Widgets
entrada = widgets.Text(
    placeholder='Escribí tu pregunta sobre las ventas...',
    layout=widgets.Layout(width='70%')
)
boton = widgets.Button(description='Enviar', button_style='primary')
salida = widgets.Output()

def mostrar_mensaje(rol, texto):
    if rol == 'usuario':
        color = '#DCF8C6'
        alineacion = 'right'
        nombre = '🧑 Vos'
    else:
        color = '#F1F0F0'
        alineacion = 'left'
        nombre = '🤖 Agente RAG'

    with salida:
        display(HTML(f"""
        <div style='text-align:{alineacion}; margin:8px'>
            <small><b>{nombre}</b></small><br>
            <span style='background:{color}; padding:8px 12px; border-radius:10px;
                         display:inline-block; max-width:80%; text-align:left'>
                {texto}
            </span>
        </div>
        """))

def enviar(b):
    pregunta = entrada.value.strip()
    if not pregunta:
        return

    entrada.value = ''
    mostrar_mensaje('usuario', pregunta)
    historial.append({'rol': 'usuario', 'texto': pregunta})

    with salida:
        display(HTML("<div style='text-align:left; margin:8px; color:gray'>⏳ Pensando...</div>"))

    try:
        respuesta = cadena_rag.invoke(pregunta)
        mostrar_mensaje('agente', respuesta)
        historial.append({'rol': 'agente', 'texto': respuesta})
    except Exception as e:
        mostrar_mensaje('agente', f'⚠️ Error: {e}')

boton.on_click(enviar)

print('💬 Chat interactivo con RAG — escribí tu pregunta y presioná Enviar')
display(widgets.HBox([entrada, boton]))
display(salida)